In [1]:
import pandas as pd
from pandas import CategoricalDtype
import geopandas as gpd
import json
import os

#show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Some preprocessing based on your chosen metro
First, some functions we'll need later on

In [ ]:
#calculate the share each occupation contributes to the total employment in the state and metro
def get_employment_rate(df,statewide=False):
    total_emp = df[df['occupation_code'] == '000000']['employment'].values[0]
    df['employment_rate'] = (df['employment'] / total_emp)*10_000
    if statewide:
        return df[['occupation_code','employment_rate']].rename(columns={'employment_rate': 'state_emp_rate'})
    return df


#slice out metro-level scores foryour main market metro and get similarly-sized metros for comparison
def get_similarly_sized_metros(metro):
    total_emp = metro_scores.loc[metro_scores['occupation_code'] == '000000']
    total_emp = total_emp.sort_values('employment', ascending=False).reset_index(drop=True)
    #get the 2 metros above and below our selected metro
    selected_metro_index = total_emp[total_emp['area_name'] == metro].index[0]
    #if selected metro is the top metro, grab 4 below
    if selected_metro_index == 0:
        similar_metros = total_emp[selected_metro_index:selected_metro_index+5]
    #if it's number 2, grab the one above and then 3 below
    elif selected_metro_index == 1:
        similar_metros = total_emp[selected_metro_index-1:selected_metro_index+4]
    #if it's the bottom one, grab 4 above
    elif selected_metro_index == len(total_emp) - 1:
        similar_metros = total_emp[selected_metro_index-4:selected_metro_index+1]
    #if it's the second to last, grab 3 above and one below
    elif selected_metro_index == len(total_emp) - 2:
        similar_metros = total_emp[selected_metro_index-3:selected_metro_index+2]
    #otherwise grab two above and two below
    else:
        similar_metros = total_emp.iloc[max(0, selected_metro_index-2):selected_metro_index+3]
    
    #remove the actual key metro from the list of similars
    similar_metros = similar_metros[similar_metros['area_name'] != metro]
    
    return similar_metros['area_name'].tolist()

def print_unique_occupations(your_main_metro, your_similar_metros_list, show_similar=False, filters=None, sort_col='employment', list_size=5):

    unique_to_here = []
    found_in_similar = []

    if filters == 'high ai, high employment':
        your_main_metro = your_main_metro[(your_main_metro['ai_exposure_category'] == 'High')&(your_main_metro['employment_category'].isin(['High','Medium high']))]
    elif filters == 'highish ai, high employment':
        your_main_metro = your_main_metro[(your_main_metro['ai_exposure_category'].isin(['High','Medium high']))&(your_main_metro['employment_category'].isin(['High','Medium high']))]
        
    occs = your_main_metro.sort_values(sort_col, ascending=False)[med_cols].head(list_size)['occupation_name']
    main_metro_occs = your_main_metro['occupation_name'].values
    
    similar_metro_occs = []
    similar_metro_dfs_list = []
    for metro_area in your_similar_metros_list:
            if filters is None:
                your_similar_metros = metro_scores[metro_scores['area_name'] == metro_area]
            elif filters == 'high ai, high employment':
                your_similar_metros = metro_scores[metro_scores['area_name'] == metro_area]
                your_similar_metros = your_similar_metros[(your_similar_metros['ai_exposure_category'] == 'High')&(your_similar_metros['employment_category'].isin(['High','Medium high']))]
            elif filters == 'highish ai, high employment':
                your_similar_metros = metro_scores[metro_scores['area_name'] == metro_area]
                your_similar_metros = your_similar_metros[(your_similar_metros['ai_exposure_category'].isin(['High','Medium high']))&(your_similar_metros['employment_category'].isin(['High','Medium high']))]
            this_metro_top = your_similar_metros.sort_values(sort_col, ascending=False)[min_cols].head(list_size)
            if show_similar:
                similar_metro_dfs_list.append(this_metro_top)
            this_metro_top_occs = this_metro_top['occupation_name'].values
            similar_metro_occs.extend(this_metro_top_occs)
    similar_metro_dfs = pd.concat(similar_metro_dfs_list)
    
    print('There are ' + str(len(your_main_metro)) + ' occupations in the main metro after applying filters. Here are the top ' + str(list_size) + ' occupations based on ' + sort_col + ':')
    for occ in occs:
        if occ in set(similar_metro_occs):
            found_in_similar.append(occ)
        else:
            unique_to_here.append(occ)
        
        jobs = your_main_metro[your_main_metro['occupation_name'] == occ]['employment'].values[0]
        jobs_formatted = "{:,}".format(int(jobs)) if jobs > 0 else '0'
        job_rate = your_main_metro[your_main_metro['occupation_name'] == occ]['employment_rate'].values[0]
        #format thousands and one decimal place
        job_rate_formatted = "{:,.1f}".format(float(job_rate))
        state_rate = your_main_metro[your_main_metro['occupation_name'] == occ]['state_emp_rate'].values[0]
        state_rate_formatted = "{:,.1f}".format(float(state_rate))
        state_compare = '🟢⬆️' if job_rate > state_rate else '🔴⬇️'
        ai_rating = your_main_metro[your_main_metro['occupation_name'] == occ]['study_rating'].values[0]
        ai_rating_formatted = "{:,.2f}".format(float(ai_rating))
        #format for thousands and convert to int for easier reading
        print('')
        print(f"{occ}")
        print(f"{jobs_formatted} workers")
        print(f"{job_rate_formatted} jobs per 10k workers vs. statewide rate of {state_rate_formatted} ({state_compare})")
        print(f"{ai_rating_formatted} AI exposure rating")
                
    if len(unique_to_here) > 0:
        print(f"Occupations in top {list_size} based on {sort_col} of main metro that aren't in top {list_size} of any similar metros: {set(unique_to_here)}")
    else:
        print(f"All occupations in the top {list_size} based on {sort_col} of the main metro are also in the top {list_size} of at least one similar metro.")
        
    if len(similar_metro_dfs) > 0:
        print(f"Top {list_size} occupations based on {sort_col} for similar metros:")
        display(similar_metro_dfs)

And now import our data and slice up into dfs for each type of geography

In [ ]:
#import our joined data
dtypes = {'series_id':str, 'year':int, 'period':str, 'areatype_code':str, 'state_code':str,
       'area_code':str, 'area_name':str, 'occupation_code':str, 'occupation_name':str,
       'employment':float, 'study_rating':float, 'footnote_codes':str, 'is_all_occupations':bool,
       'has_released_employment':bool, 'NEM Code':str, 'nem_merge':str,
       'ai_exposure_category':str, 'employment_category':str}
oews_with_scores = pd.read_csv('../data/processed/bls_occ_employment_w_study_scores_human_rating_beta.csv', dtype=dtypes)

#make sure the ai_exposure_category and employment_category use the size_order category type
size_order = CategoricalDtype(categories=['Low', 'Medium low', 'Medium high', 'High'], ordered=True)
oews_with_scores['ai_exposure_category'] = oews_with_scores['ai_exposure_category'].astype(size_order)
oews_with_scores['employment_category'] = oews_with_scores['employment_category'].astype(size_order)

#split into geo data types
metro_scores = oews_with_scores.loc[oews_with_scores['areatype_code'] == 'M']
state_scores = oews_with_scores.loc[oews_with_scores['areatype_code'] == 'S']
national_scores = oews_with_scores.loc[oews_with_scores['areatype_code'] == 'N']

In [4]:
#use this to find exact spelling of your market metro name
metro_scores.loc[metro_scores['area_name'].str.contains('San Francisco')]['area_name'].unique()

<StringArray>
['San Francisco-Oakland-Fremont, CA']
Length: 1, dtype: str

Here's where we establish which market we're interested in looking at right now. This code block is down here instaed of up higher so you can identify the exact naming of your metro in the data.

In [5]:
##########
# CHANGE FOR YOUR MARKET
##########
market_state = 'California'
market_state_abbr = 'CA'
market_metro = 'San Francisco-Oakland-Fremont, CA'
market_metro_abbr = 'sfc'
custom_metro_list = []

A last bit o' processing before we start answering questions. Basically creating different dfs that will hold:
- data for metros in our market state
- data for our main metro
- a list of similary sized metros from around the country, based on total employment counts
- data for our state as a whole
- additional information about share of jobs in each occupation 

In [6]:
#make an output folder for our state if it doesn't already exist
os.makedirs(f'../data/output/{market_state_abbr.lower()}', exist_ok=True)

#slice out metro-level scores for metros that are in your market state
market_metro_scores = metro_scores[metro_scores['area_name'].str.contains(', '+market_state_abbr)]
if len(custom_metro_list)>0:
    print('found custom metros')
    custom_metro_scores = metro_scores[metro_scores['area_name'].isin(custom_metro_list)]
    market_metro_scores = pd.concat([market_metro_scores, custom_metro_scores]).drop_duplicates()
market_metro_scores = market_metro_scores.sort_values('employment', ascending=False)
market_metro_list = market_metro_scores['area_name'].unique().tolist()

#also going to create a list of the top metros in Texas to see if anything comes of those comparisons
top_texas_metros = ['Dallas-Fort Worth-Arlington, TX',
                    'Austin-Round Rock-San Marcos, TX',
                    'San Antonio-New Braunfels, TX',
                    'El Paso, TX']

#slice out the metro-level scores for your main market metro and get similarly-sized metros for comparison
main_metro_scores = market_metro_scores[market_metro_scores['area_name'] == market_metro]
similar_metros = get_similarly_sized_metros(market_metro)

#slice out the state-level scores for your market state
market_state_scores = state_scores[state_scores['area_name'] == market_state]

#get employment shares for each occupation in the state and metro
market_state_shares = get_employment_rate(market_state_scores,statewide=True)
metro_markets_dfs = []
print('TAKE A LOOK AT THESE TO MAKE SURE THERE AREN\'T ANY WEIRD ONES IN THERE')
for metro in market_metro_list:
    #printing so we can spot anything weird since our matches are kinda squishy
    print(metro)
    metro_join = get_employment_rate(metro_scores[metro_scores['area_name'] == metro])
    metro_state_join = metro_join.merge(market_state_shares, on='occupation_code', how='left')
    metro_state_join['state_metro_diff'] = metro_state_join['employment_rate'] - metro_state_join['state_emp_rate']
    metro_markets_dfs.append(metro_state_join)
market_metro_scores = pd.concat(metro_markets_dfs)

#and just redefine the main metro scores with the employment shares and differences for easier use later
main_metro_scores = market_metro_scores[market_metro_scores['area_name'] == market_metro]

TAKE A LOOK AT THESE TO MAKE SURE THERE AREN'T ANY WEIRD ONES IN THERE
Los Angeles-Long Beach-Anaheim, CA
San Francisco-Oakland-Fremont, CA
Riverside-San Bernardino-Ontario, CA
San Diego-Chula Vista-Carlsbad, CA
San Jose-Sunnyvale-Santa Clara, CA
Sacramento-Roseville-Folsom, CA
Fresno, CA
Bakersfield-Delano, CA
Oxnard-Thousand Oaks-Ventura, CA
Stockton-Lodi, CA
Santa Rosa-Petaluma, CA
Santa Maria-Santa Barbara, CA
Modesto, CA
Salinas, CA
Visalia, CA
Vallejo, CA
San Luis Obispo-Paso Robles, CA
Santa Cruz-Watsonville, CA
Merced, CA
Chico, CA
Napa, CA
Redding, CA
El Centro, CA
Yuba City, CA
Hanford-Corcoran, CA


In [7]:
main_metro_scores.loc[main_metro_scores['occupation_code'].str.contains('3111', case=False)]

,series_id,year,period,areatype_code,state_code,area_code,area_name,occupation_code,occupation_name,employment,study_rating,footnote_codes,is_all_occupations,has_released_employment,NEM Code,nem_merge,ai_exposure_category,employment_category,top_occ_code,top_occ_name,employment_rate,state_emp_rate,state_metro_diff
16,OEUM004186000000011311101,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",113111,Compensation and Benefits Managers,590.0,0.460526,NaN,False,True,11-3111,113111,Medium low,Medium low,110000,Management Occupations,2.485341,1.630641,0.854700
49,OEUM004186000000013111101,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",131111,Management Analysts,23560.0,0.500000,NaN,False,True,13-1111,131111,Medium low,High,130000,Business and Financial Operations Occupations,99.245130,75.371835,23.873295
333,OEUM004186000000031112001,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",311120,Home Health and Personal Care Aides,119120.0,0.038462,NaN,False,True,31-1120,311120,Low,High,310000,Healthcare Support Occupations,501.786075,532.648501,-30.862426
334,OEUM004186000000031113101,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",311131,Nursing Assistants,11790.0,0.135593,NaN,False,True,31-1131,311131,Low,High,310000,Healthcare Support Occupations,49.664690,60.427041,-10.762352
335,OEUM004186000000031113201,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",311132,Orderlies,480.0,0.046875,NaN,False,True,31-1132,311132,Low,Medium low,310000,Healthcare Support Occupations,2.021972,2.673811,-0.651839


## Questions to answer:
- Which metros are similar to mine?
- Which occupations in my market have the highest employment numbers?
- Which occupations in my market have the highest AI exposure score?
- Which occupations have the worst combination of high AI exposure and large employment?
- What are the occupations in this metro that have a higher share of jobs than the state/nation?
- What share of total jobs in the metro have high exposure? Medium? Low?
- How does my metro's share of AI-exposed jobs compare to similarly-sized metros?
- Which top-level occupations (health care, service industry, etc) are most exposed in my metro? Statewide? Nationwide?

### What is "high" for AI exposure? Top is 1 (100% of tasks) so let's segment in 4 pieces:
- 0 - .25 = low
- .26 - .5 = medium low
- .51 - .75 = medium high
- .76 - 1 = high

### And what is "large" for exployment? 
Should this be relative to all of the occupations? Like quartiles or whatever? Probably. Let's see how that looks though. We might wanna slice it a different way.
- 1st quartile = low
- 2nd quartile = medium low
- 3rd quartile = medium high
- 4th quartile = high

In [8]:
#adjust as you need/want to show more/fewer important columns in your summary anaylsis below
min_cols = ['area_name','occupation_name','employment','study_rating', 
            'ai_exposure_category','employment_category']
med_cols = min_cols + ['employment_rate','state_emp_rate','state_metro_diff','top_occ_name']

## Which metros are similar to mine?

In [9]:
similar_metro_total_emp = metro_scores[((metro_scores['area_name'].isin(similar_metros))|(metro_scores['area_name'] == market_metro))&(metro_scores['occupation_code'] == '000000')]
similar_metro_total_emp.sort_values('employment', ascending=False)[['area_name','employment']]

,area_name,employment
17705,"Boston-Cambridge-Newton, MA-NH",2703890.0
130287,"Phoenix-Mesa-Chandler, AZ",2375760.0
146222,"San Francisco-Oakland-Fremont, CA",2373920.0
151076,"Seattle-Tacoma-Bellevue, WA",2086210.0
101036,"Minneapolis-St. Paul-Bloomington, MN-WI",1951650.0


## Which occupations in my market have the highest employment numbers?

In [17]:
main_metro_scores.sort_values('employment', ascending=False).head(6)

,series_id,year,period,areatype_code,state_code,area_code,area_name,occupation_code,occupation_name,employment,study_rating,footnote_codes,is_all_occupations,has_released_employment,NEM Code,nem_merge,ai_exposure_category,employment_category,top_occ_code,top_occ_name,employment_rate,state_emp_rate,state_metro_diff
0,OEUM004186000000000000001,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",000000,All Occupations,2373920.0,NaN,NaN,True,True,NaN,NaN,NaN,High,0,All Occupations,10000.000000,10000.000000,0.000000
445,OEUM004186000000043000001,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",430000,Office and Administrative Support Occupations,226320.0,NaN,NaN,False,True,NaN,NaN,NaN,High,430000,Office and Administrative Support Occupations,953.359844,1031.393951,-78.034107
1,OEUM004186000000011000001,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",110000,Management Occupations,223100.0,NaN,NaN,False,True,NaN,NaN,NaN,High,110000,Management Occupations,939.795781,730.373290,209.422490
38,OEUM004186000000013000001,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",130000,Business and Financial Operations Occupations,207030.0,NaN,NaN,False,True,NaN,NaN,NaN,High,130000,Business and Financial Operations Occupations,872.101840,694.537628,177.564212
370,OEUM004186000000035000001,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",350000,Food Preparation and Serving Related Occupations,195610.0,NaN,NaN,False,True,NaN,NaN,NaN,High,350000,Food Preparation and Serving Related Occupations,823.995754,891.065517,-67.069763
427,OEUM004186000000041000001,2025,A01,M,6,0041860,"San Francisco-Oakland-Fremont, CA",410000,Sales and Related Occupations,176900.0,NaN,NaN,False,True,NaN,NaN,NaN,High,410000,Sales and Related Occupations,745.180967,798.668036,-53.487069


In [18]:

emp_cols = ['area_name','occupation_name','employment',
            'employment_rate','state_emp_rate','study_rating']

for occ in main_metro_scores.loc[~main_metro_scores['occupation_code'].str.endswith('0000')].sort_values('employment', ascending=False)[emp_cols].head(6)['occupation_name']:
        jobs = main_metro_scores[main_metro_scores['occupation_name'] == occ]['employment'].values[0]
        jobs_formatted = "{:,}".format(int(jobs))
        job_rate = main_metro_scores[main_metro_scores['occupation_name'] == occ]['employment_rate'].values[0]
        #format thousands and one decimal place
        job_rate_formatted = "{:,.1f}".format(float(job_rate))
        state_rate = main_metro_scores[main_metro_scores['occupation_name'] == occ]['state_emp_rate'].values[0]
        state_rate_formatted = "{:,.1f}".format(float(state_rate))
        state_compare = '🟢⬆️' if job_rate > state_rate else '🔴⬇️'
        ai_rating = main_metro_scores[main_metro_scores['occupation_name'] == occ]['study_rating'].values[0]
        ai_rating_formatted = "{:,.2f}".format(float(ai_rating))
        #format for thousands and convert to int for easier reading
        print('')
        print(f"{occ}")
        print(f"{jobs_formatted} workers")
        print(f"{job_rate_formatted} jobs per 10k workers vs. statewide rate of {state_rate_formatted} ({state_compare})")
        print(f"{ai_rating_formatted}")


Home Health and Personal Care Aides
119,120 workers
501.8 jobs per 10k workers vs. statewide rate of 532.6 (🔴⬇️)
0.04

Software Developers
69,030 workers
290.8 jobs per 10k workers vs. statewide rate of 156.1 (🟢⬆️)
0.45

Fast Food and Counter Workers
49,970 workers
210.5 jobs per 10k workers vs. statewide rate of 247.0 (🔴⬇️)
0.07

General and Operations Managers
43,520 workers
183.3 jobs per 10k workers vs. statewide rate of 165.2 (🟢⬆️)
0.38

Registered Nurses
41,750 workers
175.9 jobs per 10k workers vs. statewide rate of 186.1 (🔴⬇️)
0.38

Retail Salespersons
39,460 workers
166.2 jobs per 10k workers vs. statewide rate of 206.7 (🔴⬇️)
0.36


## Which occupations in my market have the highest AI exposure score?

I wouldn't expect there to be occupations unique to our main market here FYI. This is just establishing a baseline to get us used to the highest study ratings and how those present in our market.

In [11]:
emp_cols = ['area_name','occupation_name','employment',
            'employment_rate','state_emp_rate','study_rating']
for occ in main_metro_scores.sort_values('study_rating', ascending=False)[emp_cols].head(6)['occupation_name']:
        jobs = main_metro_scores[main_metro_scores['occupation_name'] == occ]['employment'].values[0]
        jobs_formatted = "{:,}".format(int(jobs)) if jobs > 0 else '0'
        job_rate = main_metro_scores[main_metro_scores['occupation_name'] == occ]['employment_rate'].values[0]
        #format thousands and one decimal place
        job_rate_formatted = "{:,.1f}".format(float(job_rate))
        state_rate = main_metro_scores[main_metro_scores['occupation_name'] == occ]['state_emp_rate'].values[0]
        state_rate_formatted = "{:,.1f}".format(float(state_rate))
        state_compare = '🟢⬆️' if job_rate > state_rate else '🔴⬇️'
        ai_rating = main_metro_scores[main_metro_scores['occupation_name'] == occ]['study_rating'].values[0]
        ai_rating_formatted = "{:,.2f}".format(float(ai_rating))
        #format for thousands and convert to int for easier reading
        print('')
        print(f"{occ}")
        print(f"{jobs_formatted} workers")
        print(f"{job_rate_formatted} jobs per 10k workers vs. statewide rate of {state_rate_formatted} ({state_compare})")
        print(f"{ai_rating_formatted}")


Survey Researchers
180 workers
0.8 jobs per 10k workers vs. statewide rate of 0.6 (🟢⬆️)
0.84

Interpreters and Translators
760 workers
3.2 jobs per 10k workers vs. statewide rate of 3.1 (🟢⬆️)
0.84

Writers and Authors
1,560 workers
6.6 jobs per 10k workers vs. statewide rate of 4.6 (🟢⬆️)
0.81

Brokerage Clerks
1,360 workers
5.7 jobs per 10k workers vs. statewide rate of 2.6 (🟢⬆️)
0.80

Public Relations Specialists
6,290 workers
26.5 jobs per 10k workers vs. statewide rate of 18.0 (🟢⬆️)
0.79

Legal Secretaries and Administrative Assistants
2,740 workers
11.5 jobs per 10k workers vs. statewide rate of 15.1 (🔴⬇️)
0.76


## Which occupations have the worst combination of high AI exposure and large employment?

In [12]:
#filters = 'high ai, high employment'
filters = 'highish ai, high employment'
print_unique_occupations(
    your_main_metro=main_metro_scores, 
    your_similar_metros_list=similar_metros, 
    show_similar=True, 
    filters=filters,
    sort_col='employment',
    list_size=5
    )

There are 67 occupations in the main metro after applying filters. Here are the top 5 occupations based on employment:

Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel
29,980 workers
126.3 jobs per 10k workers vs. statewide rate of 85.7 (🟢⬆️)
0.57 AI exposure rating

Accountants and Auditors
25,170 workers
106.0 jobs per 10k workers vs. statewide rate of 96.3 (🟢⬆️)
0.52 AI exposure rating

Market Research Analysts and Marketing Specialists
24,120 workers
101.6 jobs per 10k workers vs. statewide rate of 65.4 (🟢⬆️)
0.58 AI exposure rating

Computer and Information Systems Managers
23,830 workers
100.4 jobs per 10k workers vs. statewide rate of 52.0 (🟢⬆️)
0.56 AI exposure rating

Business Operations Specialists, All Other
23,400 workers
98.6 jobs per 10k workers vs. statewide rate of 85.1 (🟢⬆️)
0.54 AI exposure rating
Occupations in top 5 based on employment of main metro that aren't in top 5 of any similar metros: {'Computer and Informatio

,area_name,occupation_name,employment,study_rating,ai_exposure_category,employment_category
18157,"Boston-Cambridge-Newton, MA-NH",Customer Service Representatives,37780.0,0.704545,Medium high,High
17758,"Boston-Cambridge-Newton, MA-NH",Accountants and Auditors,37150.0,0.520000,Medium high,High
17756,"Boston-Cambridge-Newton, MA-NH",Market Research Analysts and Marketing Specialists,32310.0,0.576923,Medium high,High
18184,"Boston-Cambridge-Newton, MA-NH","Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",28540.0,0.567308,Medium high,High
18144,"Boston-Cambridge-Newton, MA-NH",First-Line Supervisors of Office and Administrative Support Workers,25630.0,0.531250,Medium high,High
130713,"Phoenix-Mesa-Chandler, AZ",Customer Service Representatives,68930.0,0.704545,Medium high,High
130700,"Phoenix-Mesa-Chandler, AZ",First-Line Supervisors of Office and Administrative Support Workers,23940.0,0.531250,Medium high,High
130739,"Phoenix-Mesa-Chandler, AZ","Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",22010.0,0.567308,Medium high,High
130338,"Phoenix-Mesa-Chandler, AZ",Accountants and Auditors,21360.0,0.520000,Medium high,High
130691,"Phoenix-Mesa-Chandler, AZ","Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel",19810.0,0.566667,Medium high,High


## What are the occupations in this metro that have a higher share of jobs than the state?

In [19]:
rename_cols = {'area_name':'Location', 'occupation_name':'Occupation','employment':'Jobs',
               'employment_category':'Employment category','study_rating':'AI exposure rating',
               'ai_exposure_category':'AI exposure category','employment_rate':'Jobs per 10k', 
               'state_emp_rate':'State jobs per 10k','state_metro_diff':'State-metro difference'}
more_jobs_than_state = main_metro_scores.loc[~main_metro_scores['occupation_code'].str.endswith('0000')].sort_values('state_metro_diff', ascending=False).head(10)
more_jobs_than_state.rename(columns=rename_cols, inplace=True)
display(more_jobs_than_state[rename_cols.values()])
more_jobs_than_state[rename_cols.values()].to_csv(f'../data/output/{market_state_abbr.lower()}/{market_metro_abbr}_top_10_occupations_with_larger_employment_in_metro_than_state.csv', index=False)

,Location,Occupation,Jobs,Employment category,AI exposure rating,AI exposure category,Jobs per 10k,State jobs per 10k,State-metro difference
81,"San Francisco-Oakland-Fremont, CA",Software Developers,69030.0,High,0.447368,Medium low,290.784862,156.140707,134.644155
11,"San Francisco-Oakland-Fremont, CA",Computer and Information Systems Managers,23830.0,High,0.560606,Medium high,100.382490,52.004810,48.377680
438,"San Francisco-Oakland-Fremont, CA","Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel",29980.0,High,0.566667,Medium high,126.289007,85.710207,40.578800
54,"San Francisco-Oakland-Fremont, CA",Market Research Analysts and Marketing Specialists,24120.0,High,0.576923,Medium high,101.604098,65.373867,36.230231
37,"San Francisco-Oakland-Fremont, CA","Managers, All Other",21550.0,High,0.465491,Medium low,90.778122,54.673131,36.104992
85,"San Francisco-Oakland-Fremont, CA","Computer Occupations, All Other",16930.0,High,0.596964,Medium high,71.316641,44.005337,27.311304
180,"San Francisco-Oakland-Fremont, CA",Lawyers,18470.0,High,0.475000,Medium low,77.803801,52.581299,25.222503
48,"San Francisco-Oakland-Fremont, CA",Project Management Specialists,20770.0,High,0.400000,Medium low,87.492418,62.798882,24.693535
49,"San Francisco-Oakland-Fremont, CA",Management Analysts,23560.0,High,0.500000,Medium low,99.245130,75.371835,23.873295
89,"San Francisco-Oakland-Fremont, CA",Data Scientists,10460.0,High,0.593750,Medium high,44.062142,21.582655,22.479487


## What share of total jobs in the metro have high exposure? Medium? Low?

I'm also curious how many occupations scored in the study fall into each of the ai exposure categories:

In [20]:
def get_ai_exposure_category(score):
    if score <= .25:
        return 'Low'
    elif score <= .5:
        return 'Medium low'
    elif score <= .75:
        return 'Medium high'
    elif score > .75:
        return 'High'
    else:
        return 'NA'

occ_scores = pd.read_csv('https://raw.githubusercontent.com/openai/GPTs-are-GPTs/refs/heads/main/data/occ_level.csv')
study_rating = 'human_rating_beta'
occ_scores = occ_scores[['O*NET-SOC Code', 'Title', study_rating]]
occ_scores['occupation_code'] = occ_scores['O*NET-SOC Code'].str.replace('-', '')#.str[:6]
occ_scores['occupation_code'] = occ_scores['occupation_code'].apply(lambda x: x if len(x) == 6 else x[:6])
occ_scores = occ_scores.rename(columns={study_rating: 'study_rating', 
                                        'O*NET-SOC Code': 'soc_code',
                                        'Title': 'occupation_name'})

all_occ_scores = occ_scores.copy()
all_occ_scores['ai_exposure_category'] = all_occ_scores['study_rating'].apply(get_ai_exposure_category)
# Apply the ordered category type
all_occ_scores['ai_exposure_category'] = all_occ_scores['ai_exposure_category'].astype(size_order)

all_occ_by_score = all_occ_scores.groupby('ai_exposure_category',dropna=False)['occupation_code'].count().reset_index()
all_occ_by_score['share'] = (all_occ_by_score['occupation_code'] / len(all_occ_scores))*100

print('Number of unique occupations in each AI exposure category:')
all_occ_by_score.sort_values('ai_exposure_category',ascending=False)

Number of unique occupations in each AI exposure category:


,ai_exposure_category,occupation_code,share
3,High,8,0.866739
2,Medium high,169,18.309859
1,Medium low,343,37.161430
0,Low,403,43.661972


In [24]:
total_metro_emp = metro_scores.loc[(metro_scores['occupation_code'] == '000000')&(metro_scores['area_name'] == market_metro)]
metro_by_exposure = main_metro_scores.groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
metro_by_exposure['share'] = (metro_by_exposure['employment'] / total_metro_emp['employment'].values[0])*100

print(f'Share of total metro employment in each AI exposure category for {market_metro}:')
metro_by_exposure.sort_values('ai_exposure_category',ascending=False)

Share of total metro employment in each AI exposure category for San Francisco-Oakland-Fremont, CA:


,ai_exposure_category,employment,share
3,Medium low,936490.0,39.449097
2,Medium high,471370.0,19.856187
1,Low,882960.0,37.194177
0,High,12890.0,0.542984
4,NaN,4790750.0,201.807559


I should also probably look at these buckets statewide and nationwide, yeah?

In [25]:
total_state_emp = state_scores.loc[(state_scores['occupation_code'] == '000000')&(state_scores['area_name'] == market_state)]
state_by_exposure = state_scores.loc[(state_scores['area_name'] == market_state)].groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
state_by_exposure['share'] = (state_by_exposure['employment'] / total_state_emp['employment'].values[0])*100

print(f'Share of total state employment in each AI exposure category for {market_state}:')
state_by_exposure.sort_values('ai_exposure_category',ascending=False)

Share of total state employment in each AI exposure category for California:


,ai_exposure_category,employment,share
3,Medium low,6823580.0,37.463997
2,Medium high,3059060.0,16.795379
1,Low,7816990.0,42.918188
0,High,80190.0,0.440273
4,NaN,36797130.0,202.029955


In [26]:
total_national_emp = national_scores.loc[(national_scores['occupation_code'] == '000000')]
national_by_exposure = national_scores.groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
national_by_exposure['share'] = (national_by_exposure['employment'] / total_national_emp['employment'].values[0])*100

print(f'Share of total national employment in each AI exposure category:')
national_by_exposure.sort_values('ai_exposure_category',ascending=False)

Share of total national employment in each AI exposure category:


,ai_exposure_category,employment,share
3,Medium low,59552180.0,38.298274
2,Medium high,27329870.0,17.575962
1,Low,65971620.0,42.426644
0,High,586990.0,0.377496
4,NaN,534203880.0,343.548906


In [27]:
#combine them all so we can compare better and turn into a quick chart
by_exposure = metro_by_exposure.merge(state_by_exposure, on='ai_exposure_category', how='left', suffixes=('_metro','_state'))
by_exposure = by_exposure.merge(national_by_exposure, on='ai_exposure_category', how='left')
by_exposure = by_exposure.rename(columns={'employment': 'emp_national', 'share': 'National','share_metro':'Metro','share_state':'State','share_national':'National'})
by_exposure = by_exposure[['ai_exposure_category','Metro','State','National']]
by_exposure = by_exposure.loc[by_exposure['ai_exposure_category'] != 'NA']


by_exposure.sort_values('ai_exposure_category',ascending=False).to_csv(f'../data/output/{market_state_abbr.lower()}/{market_metro_abbr}_employment_by_ai_exposure.csv', index=False)

## How does my metro's share of AI-exposed jobs compare to similarly-sized metros?

In [28]:
total_metro_emp = metro_scores.loc[(metro_scores['occupation_code'] == '000000')&(metro_scores['area_name'] == market_metro)]
metro_by_exposure = main_metro_scores.groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
metro_by_exposure['share'] = (metro_by_exposure['employment'] / total_metro_emp['employment'].values[0])*100

print(f'Share of total metro employment in each AI exposure category for {market_metro}:')
metro_by_exposure.sort_values('ai_exposure_category',ascending=False)

Share of total metro employment in each AI exposure category for San Francisco-Oakland-Fremont, CA:


,ai_exposure_category,employment,share
3,Medium low,936490.0,39.449097
2,Medium high,471370.0,19.856187
1,Low,882960.0,37.194177
0,High,12890.0,0.542984
4,NaN,4790750.0,201.807559


In [29]:
#checkout all similiarly-sized metros
for metro_area in similar_metros:
    print(metro_area)
    this_metro_scores = metro_scores[metro_scores['area_name'] == metro_area]
    total_metro_emp = this_metro_scores.loc[(this_metro_scores['occupation_code'] == '000000')]
    metro_by_exposure = this_metro_scores.groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
    metro_by_exposure['share'] = (metro_by_exposure['employment'] / total_metro_emp['employment'].values[0])*100

    print(f'Share of total metro employment in each AI exposure category for {metro_area}:')
    display(metro_by_exposure.sort_values('ai_exposure_category',ascending=False))
    # this_metro_rate = get_employment_rate(metro_scores.loc[metro_scores['area_name'] == metro_area], statewide=False)
    # display(this_metro_rate.groupby('ai_exposure_category',dropna=False)['employment_rate'].sum())

Boston-Cambridge-Newton, MA-NH
Share of total metro employment in each AI exposure category for Boston-Cambridge-Newton, MA-NH:


,ai_exposure_category,employment,share
3,Medium low,1139740.0,42.151863
2,Medium high,555580.0,20.547434
1,Low,931690.0,34.457393
0,High,12410.0,0.458968
4,NaN,5423600.0,200.585083


Phoenix-Mesa-Chandler, AZ
Share of total metro employment in each AI exposure category for Phoenix-Mesa-Chandler, AZ:


,ai_exposure_category,employment,share
3,Medium low,895650.0,37.699515
2,Medium high,443270.0,18.658029
1,Low,981220.0,41.301310
0,High,8760.0,0.368724
4,NaN,4768160.0,200.700407


Seattle-Tacoma-Bellevue, WA
Share of total metro employment in each AI exposure category for Seattle-Tacoma-Bellevue, WA:


,ai_exposure_category,employment,share
3,Medium low,813690.0,39.003264
2,Medium high,443160.0,21.242349
1,Low,781900.0,37.479448
0,High,7870.0,0.377239
4,NaN,4189710.0,200.828776


Minneapolis-St. Paul-Bloomington, MN-WI
Share of total metro employment in each AI exposure category for Minneapolis-St. Paul-Bloomington, MN-WI:


,ai_exposure_category,employment,share
3,Medium low,746560.0,38.252760
2,Medium high,357780.0,18.332180
1,Low,793080.0,40.636385
0,High,10370.0,0.531345
4,NaN,3927710.0,201.250737


In [ ]:
metro_scores.head()

## Which top-level occupations (health care, service industry, etc) are most exposed in my metro?

In [30]:
def get_exposure_job_share(group):
    return pd.Series({
        'share_high_exposure': (group[group['ai_exposure_category'] == 'High']['employment'].sum()/group['employment'].sum())*100,
        'share_medhigh_exposure': (group[group['ai_exposure_category'] == 'Medium high']['employment'].sum()/group['employment'].sum())*100,
        'share_medlow_exposure': (group[group['ai_exposure_category'] == 'Medium low']['employment'].sum()/group['employment'].sum())*100,
        'share_low_exposure': (group[group['ai_exposure_category'] == 'Low']['employment'].sum()/group['employment'].sum())*100
    })
by_top_occ_1 = main_metro_scores.groupby(['top_occ_code',
                                        'top_occ_name'],dropna=False).agg(total_jobs=('employment','sum'),
                                                                        occupations=('occupation_code','count'),
                                                                        avg_study_rating=('study_rating','mean'),
                                                                        median_study_rating=('study_rating','median')
                                                                        ).reset_index()
by_top_occ_2 = main_metro_scores.groupby(['top_occ_code',
                                          'top_occ_name'],dropna=False).apply(get_exposure_job_share).reset_index()
by_top_occ = by_top_occ_1.merge(by_top_occ_2, on=['top_occ_code','top_occ_name'], how='left')

In [31]:
rename_cols = {'top_occ_code': 'Occupation group code', 'top_occ_name': 'Occupation group', 
               'occupations': 'Occupations in group','total_jobs': 'Jobs', 
               'avg_study_rating': 'Avg. study rating', 'share_high_exposure': '% high exposure', 
               'share_medhigh_exposure': '% medium-high exposure', 
               'share_medlow_exposure': '% medium-low exposure', 'share_low_exposure': '% low exposure'}
by_top_occ.rename(columns=rename_cols, inplace=True)
by_top_occ.sort_values('% medium-high exposure', ascending=False).to_csv(f'../data/output/{market_state_abbr.lower()}/{market_metro_abbr}_by_top_occ.csv', index=False)

In [33]:
by_top_occ.sort_values('% medium-high exposure', ascending=False).head()

,Occupation group code,Occupation group,Jobs,Occupations in group,Avg. study rating,median_study_rating,% high exposure,% medium-high exposure,% medium-low exposure,% low exposure
2,130000,Business and Financial Operations Occupations,414030.0,32,0.536440,0.537990,0.000000,29.633118,20.363259,0.000000
17,430000,Office and Administrative Support Occupations,452550.0,50,0.469441,0.485179,0.905977,25.530881,21.688211,0.724782
3,150000,Computer and Mathematical Occupations,311920.0,20,0.584828,0.596774,0.000000,23.733650,26.250321,0.000000
16,410000,Sales and Related Occupations,350920.0,18,0.504761,0.524767,0.000000,20.802462,27.929443,0.000000
5,190000,"Life, Physical, and Social Science Occupations",76300.0,38,0.485591,0.500000,0.235911,20.052425,25.806029,0.432503
